*  Course Name: DSC 550
*  Assignment Name: Week 6 Coding Assignment
*  Student Name: Anish Samuel

In [72]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier

# Part 1: PCA and Variance Threshold in a Linear Regression

In [73]:
# Import the housing data as a data frame and ensure that the data is loaded properly.
df = pd.read_csv('train.csv')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [74]:
# Drop the "Id" column and any features that are missing more than 40% of their values.
threshold = 0.4 * len(df)
df = df.drop(columns=['Id'])
df = df.dropna(thresh=threshold, axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 76 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   LotShape       1460 non-null   object 
 6   LandContour    1460 non-null   object 
 7   Utilities      1460 non-null   object 
 8   LotConfig      1460 non-null   object 
 9   LandSlope      1460 non-null   object 
 10  Neighborhood   1460 non-null   object 
 11  Condition1     1460 non-null   object 
 12  Condition2     1460 non-null   object 
 13  BldgType       1460 non-null   object 
 14  HouseStyle     1460 non-null   object 
 15  OverallQual    1460 non-null   int64  
 16  OverallCond    1460 non-null   int64  
 17  YearBuilt      1460 non-null   int64  
 18  YearRemo

In [75]:
# For numerical columns, fill in any missing data with the median value.
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)


/var/folders/np/trb8r5595yx20_037znb610r0000gn/T/ipykernel_41289/492042237.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [76]:
# For categorical columns, fill in any missing data with the most common value (mode).
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

/var/folders/np/trb8r5595yx20_037znb610r0000gn/T/ipykernel_41289/2753962402.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [77]:
#Convert categorical columns to dummy variables
df = pd.get_dummies(df, drop_first=True)
# Split the data into features and target variable
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Run a linear regression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
from sklearn.metrics import mean_squared_error, root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)
print(f'R2: {r2:.4f}, RMSE: {rmse:.4f}')
print(sklearn.__version__)

R2: 0.6434, RMSE: 52298.8715
1.6.1


In [78]:
# Fit and transform the training features with a PCA so that 90% of the variance is retained
pca = PCA(n_components=0.9)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
print(f'Number of PCA features: {X_train_pca.shape[1]}')

Number of PCA features: 1


In [79]:
# Repeat the linear regression with the PCA transformed data
model_pca = LinearRegression()
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)
r2_pca = r2_score(y_test, y_pred_pca)
rmse_pca = root_mean_squared_error(y_test, y_pred_pca)
print(f'PCA R2: {r2_pca:.4f}, PCA RMSE: {rmse_pca:.4f}')

PCA R2: 0.0635, PCA RMSE: 84754.5802


In [80]:
# apply a min-max scaler to orginal set
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Find the min-max scaled features in your training set that have a variance above 0.1 
high_variance_features = np.var(X_train_scaled, axis=0) > 0.1
X_train_high_variance = X_train_scaled[:, high_variance_features]
X_test_high_variance = X_test_scaled[:, high_variance_features]
# Repeat with the high variance data
model_high_variance = LinearRegression()
model_high_variance.fit(X_train_high_variance, y_train)
y_pred_high_variance = model_high_variance.predict(X_test_high_variance)
r2_high_variance = r2_score(y_test, y_pred_high_variance)
rmse_high_variance = root_mean_squared_error(y_test, y_pred_high_variance)
print(f'High Variance R2: {r2_high_variance:.4f}, High Variance RMSE: {rmse_high_variance:.4f}')

High Variance R2: 0.6556, High Variance RMSE: 51393.4322


In [81]:
# Summarize your findings
print("Summary of Findings:")
print(f"Original R2: {r2:.4f}, RMSE: {rmse:.4f}")
print(f"PCA R2: {r2_pca:.4f}, RMSE: {rmse_pca:.4f}")
print(f"High Variance R2: {r2_high_variance:.4f}, RMSE: {rmse_high_variance:.4f}")
# The PCA transformation reduced the number of features while retaining 90% of the variance, which can lead to better model performance.
# The high variance features after min-max scaling also provided a good performance, indicating that focusing on features with higher variance can be beneficial.
# The original model performed well, but the PCA and high variance transformations provided alternative approaches that can be useful depending on the dataset characteristics.

Summary of Findings:
Original R2: 0.6434, RMSE: 52298.8715
PCA R2: 0.0635, RMSE: 84754.5802
High Variance R2: 0.6556, RMSE: 51393.4322


# Part 2: Categorical Feature Selection

In [83]:
mushroom_df = pd.read_csv('mushrooms.csv')
mushroom_df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [84]:
# Convert categorical features to dummy variables
mushroom_df = pd.get_dummies(mushroom_df, drop_first=True)

# Split the data into features and target variable
X_mushroom = mushroom_df.drop(columns=['class_p'])
y_mushroom = mushroom_df['class_p']
# Split the data into training and test sets
X_train_mushroom, X_test_mushroom, y_train_mushroom, y_test_mushroom = train_test_split(X_mushroom, y_mushroom, test_size=0.2, random_state=42)
# Fit a decision tree classifier on the training set

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_mushroom, y_train_mushroom)


DecisionTreeClassifier(random_state=42)

In [85]:
# Report the accuracy and create a confusion matrix for the model prediction on the test set
from sklearn.metrics import accuracy_score, confusion_matrix
y_pred_mushroom = dt_model.predict(X_test_mushroom)
accuracy = accuracy_score(y_test_mushroom, y_pred_mushroom)
conf_matrix = confusion_matrix(y_test_mushroom, y_pred_mushroom)
print(f'Accuracy: {accuracy:.4f}')
print('Confusion Matrix:\n', conf_matrix)


Accuracy: 1.0000
Confusion Matrix:
 [[843   0]
 [  0 782]]


In [90]:
# Create a visualization of the decision tree
from sklearn.tree import export_graphviz
import graphviz
dot_data = export_graphviz(dt_model, out_file=None, 
                           feature_names=X_train_mushroom.columns,  
                           class_names=['edible', 'poisonous'],  
                           filled=True, rounded=True,  
                           special_characters=True)
graph = graphviz.Source(dot_data)
# display graph
graph.render("mushroom_decision_tree", format='png', cleanup=True)
# display the graph here
graph.view()

'mushroom_decision_tree.pdf'

In [87]:
# Use a χ2-statistic selector to pick the five best features
from sklearn.feature_selection import SelectKBest, chi2
selector = SelectKBest(score_func=chi2, k=5)
selector.fit(X_train_mushroom, y_train_mushroom)
best_features = selector.get_support(indices=True)
best_feature_names = X_train_mushroom.columns[best_features]
print("Best Features Selected:", best_feature_names.tolist())

Best Features Selected: ['odor_f', 'odor_n', 'gill-size_n', 'stalk-surface-above-ring_k', 'stalk-surface-below-ring_k']


In [88]:
# Repeat steps 4 and 5 with the five best features selected
X_train_best = X_train_mushroom.iloc[:, best_features]
X_test_best = X_test_mushroom.iloc[:, best_features]
dt_model_best = DecisionTreeClassifier(random_state=42)
dt_model_best.fit(X_train_best, y_train_mushroom)
y_pred_best = dt_model_best.predict(X_test_best)
accuracy_best = accuracy_score(y_test_mushroom, y_pred_best)
conf_matrix_best = confusion_matrix(y_test_mushroom, y_pred_best)
print(f'Best Features Model Accuracy: {accuracy_best:.4f}')
print('Best Features Confusion Matrix:\n', conf_matrix_best)

Best Features Model Accuracy: 0.9735
Best Features Confusion Matrix:
 [[816  27]
 [ 16 766]]


In [89]:
# Summarize your findings
print("Summary of Findings for Mushroom Classification:")
print(f"Initial Model Accuracy: {accuracy:.4f}")
print(f"Best Features Model Accuracy: {accuracy_best:.4f}")
print("Best Features Selected:", best_feature_names.tolist())
# The decision tree classifier performed well on the mushroom dataset, achieving a high accuracy.
# The confusion matrix indicates the model's performance in classifying edible and poisonous mushrooms.
# The feature selection using the χ2-statistic helped identify the five most significant features, which improved the model's performance.
# The visualization of the decision tree provides insights into how the model makes decisions based on the features.

Summary of Findings for Mushroom Classification:
Initial Model Accuracy: 1.0000
Best Features Model Accuracy: 0.9735
Best Features Selected: ['odor_f', 'odor_n', 'gill-size_n', 'stalk-surface-above-ring_k', 'stalk-surface-below-ring_k']
